# 09 — Ablaciones: recuperación biológica con el score bloqueado 0.075

Reconstruye los grafos de **FULL y de las cuatro ablaciones atómicas ya entrenadas** con la puntuación biológica definitiva

\[R(|d|)+0.35R(\operatorname{sign}(d)q)+0.075R(f).\]

No reentrena. Para cada brazo conserva Exact-K=529 en las cinco semillas, redefine el núcleo dirigido 5/5, cuantifica estabilidad y componentes, cruza el núcleo con las cuatro firmas bloqueadas, repite el nulo emparejado para Basal-like y ejecuta la misma ORA Hallmark/C2 del notebook 05.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'intergate').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Ejecute este notebook dentro del repositorio InterGATE.')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_transport_protocol, assert_v6_protocol, prepare_runtime,
)
from intergate.posthoc_ablation_biology import (
    run_locked_score_ablation_biology,
)

assert_v6_protocol(CFG)
assert_transport_protocol(CFG)

In [ ]:
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RESULTS = PROJECT_ROOT / 'results_v6'
RESOURCE_DIR = PROJECT_ROOT / 'resources_biological'
PROTOCOL_LOCK = ARTIFACTS / 'protocol_lock_v6.json'
OUT = RESULTS / '09_ablation_biology_w0075'

N_MATCHED_DRAWS = 1000
RANDOM_SEED = 20260821

required_artifacts = [
    'INTERGATE_V6',
    'INTERGATE_V6_ABL_NO_CONTEXT_LOSS',
    'INTERGATE_V6_ABL_NO_COHORT_QUERY_LOSS',
    'INTERGATE_V6_ABL_NO_EDGE_TYPE_CALIBRATION',
    'INTERGATE_V6_ABL_NO_SIGNED_WEIGHT',
]
missing = [name for name in required_artifacts if not (ARTIFACTS / name).is_dir()]
if missing:
    raise FileNotFoundError('Faltan brazos entrenados del notebook 04: ' + ', '.join(missing))
print('Se reutilizarán cinco brazos × cinco semillas; entrenamientos nuevos: 0')
print('Salida:', OUT)

In [ ]:
# Runtime exclusivamente de desarrollo: no se leen matrices ni etiquetas externas.
runtime = prepare_runtime(CFG, include_external=False)
assert runtime['external_opened'] is False

ablation = run_locked_score_ablation_biology(
    runtime,
    artifacts_root=ARTIFACTS,
    results_root=RESULTS,
    resource_dir=RESOURCE_DIR,
    output_dir=OUT,
    protocol_lock=PROTOCOL_LOCK,
    n_matched_draws=N_MATCHED_DRAWS,
    random_seed=RANDOM_SEED,
)
print('Post-procesamiento biológico de ablaciones completo.')

In [ ]:
summary = ablation['summary'].copy()
display(summary[[
    'arm', 'mean_pairwise_edge_jaccard', 'strict_edges', 'strict_nodes',
    'strict_n_components', 'strict_giant_nodes', 'strict_giant_fraction',
    'strict_edge_jaccard_vs_full', 'basal_specificity_margin',
    'basal_fold_over_null__at_least_one_endpoint',
    'basal_fold_over_null__both_endpoints',
    'basal_fold_over_null__unique_node_fraction',
]])

display(ablation['matched_tests'][[
    'arm', 'statistic', 'observed', 'null_mean', 'fold_over_null',
    'empirical_p', 'fdr_global_15_tests',
]])

display(ablation['paired_tests'])

In [ ]:
# Vías significativas por brazo; la tabla completa conserva todos los términos.
ora_sig = ablation['ora_significant'].copy()
display(ora_sig.sort_values(['arm', 'collection', 'fdr']).groupby(
    ['arm', 'collection'], group_keys=False
).head(12)[[
    'arm', 'collection', 'term', 'overlap_n', 'fold_enrichment',
    'fdr', 'fdr_global_all_arms_same_collection',
]])

In [ ]:
# Resumen visual centrado en estructura y recuperación, no en predicción.
plot = summary.set_index('arm')
order = [
    'FULL', 'NO_CONTEXT_LOSS', 'NO_COHORT_QUERY_LOSS',
    'NO_EDGE_TYPE_CALIBRATION', 'NO_SIGNED_WEIGHT',
]
labels = ['FULL', 'sin contexto', 'sin query de cohorte', 'sin tipo', 'sin signo']
plot = plot.loc[order]
colors = ['#2166ac'] + ['#b2182b'] * 4
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
metrics = [
    ('mean_pairwise_edge_jaccard', 'Jaccard medio entre semillas'),
    ('strict_edges', 'Aristas del núcleo 5/5'),
    ('basal_fold_over_null__unique_node_fraction',
     'Recuperación Basal-like: fold sobre nulo'),
    ('strict_giant_fraction', 'Fracción de nodos en componente mayor'),
]
for ax, (column, title) in zip(axes.flat, metrics):
    ax.bar(np.arange(len(plot)), plot[column], color=colors)
    ax.set_xticks(np.arange(len(plot)), labels, rotation=28, ha='right')
    ax.set_title(title)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Ablaciones atómicas: estructura y recuperación biológica', y=1.01)
fig.tight_layout()
fig.savefig(OUT / 'ablation_biology_w0075.pdf', bbox_inches='tight')
fig.savefig(OUT / 'ablation_biology_w0075.png', dpi=600, bbox_inches='tight')
plt.show()

## Regla de interpretación

La evidencia principal será estructural y biológica: estabilidad entre semillas, tamaño y organización del núcleo dirigido 5/5, margen de especificidad Basal-like, enriquecimiento frente al nulo emparejado y conservación de programas Hallmark/C2. Las pruebas pareadas de cinco semillas se consideran estimaciones de efecto con potencia limitada: su menor valor p bilateral posible es 0.0625. Las métricas predictivas del notebook 04 no se reutilizan para decidir qué componente produce mejor recuperación.